In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_transactions.csv")
df.head()

,transaction_id,timestamp,transaction_type,merchant_category,amount_inr,transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [2]:
print(df["fraud_flag"].value_counts())
print(df["fraud_flag"].mean())

fraud_flag
0    249520
1       480
Name: count, dtype: int64
0.00192


In [4]:
rules = pd.DataFrame(index=df.index)

# Derive a plain date from timestamp for grouping (if a dedicated date column doesn't exist)
df["timestamp"] = pd.to_datetime(df["timestamp"])
txn_date = df["timestamp"].dt.date

# Rule 1: high transaction velocity proxy (state + calendar date + hour)
state_hour_counts = df.groupby([df["sender_state"], txn_date, df["hour_of_day"]])["transaction_id"].transform("count")
rules["high_velocity_segment"] = state_hour_counts > state_hour_counts.quantile(0.99)

# Rule 2: unusually high amount relative to the transaction's own category
category_p99 = df.groupby("merchant_category")["amount_inr"].transform(lambda x: x.quantile(0.99))
rules["amount_above_category_p99"] = df["amount_inr"] > category_p99

# Rule 3: odd hour (very late night / early morning)
rules["odd_hour"] = df["hour_of_day"].isin([1,2,3,4])

# Combine: flagged if 2 or more rules trip simultaneously
rules["rule_flags_triggered"] = rules[["high_velocity_segment","amount_above_category_p99","odd_hour"]].sum(axis=1)
rules["rule_based_flag"] = rules["rule_flags_triggered"] >= 2

rules["rule_based_flag"].value_counts()

rule_based_flag
False    249921
True         79
Name: count, dtype: int64

In [6]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

feature_cols = ["amount_inr", "hour_of_day"]
X = df[feature_cols].copy()

# one-hot encode a couple of categoricals so IsolationForest can use them
X = pd.get_dummies(df[feature_cols + ["merchant_category", "device_type", "network_type"]], 
                    columns=["merchant_category", "device_type", "network_type"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.0019,  # matches our known fraud rate (480/250000) - reasonable prior
    random_state=42
)

df["anomaly_score"] = iso_forest.fit_predict(X_scaled)
df["is_anomaly"] = df["anomaly_score"] == -1

df["is_anomaly"].value_counts()

is_anomaly
False    249525
True        475
Name: count, dtype: int64

In [8]:
from sklearn.metrics import classification_report

df["rule_based_flag"] = rules["rule_based_flag"].values

print("=== Rule-based flags vs actual fraud ===")
print(classification_report(df["fraud_flag"], df["rule_based_flag"]))

print("=== Isolation Forest vs actual fraud ===")
print(classification_report(df["fraud_flag"], df["is_anomaly"].astype(int)))

=== Rule-based flags vs actual fraud ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    249520
           1       0.00      0.00      0.00       480

    accuracy                           1.00    250000
   macro avg       0.50      0.50      0.50    250000
weighted avg       1.00      1.00      1.00    250000

=== Isolation Forest vs actual fraud ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    249520
           1       0.00      0.00      0.00       480

    accuracy                           1.00    250000
   macro avg       0.50      0.50      0.50    250000
weighted avg       1.00      1.00      1.00    250000



In [9]:
print(df["fraud_flag"].dtype, df["fraud_flag"].unique())
print(df["rule_based_flag"].dtype, df["rule_based_flag"].unique())
print(df["is_anomaly"].dtype, df["is_anomaly"].unique())

# Confirm index alignment wasn't silently broken anywhere
print(len(df), len(rules), (df.index == rules.index).all())

int64 [0 1]
bool [False  True]
bool [False  True]
250000 250000 True


In [10]:
n_rule_flagged = df["rule_based_flag"].sum()
n_iso_flagged = df["is_anomaly"].sum()
n_fraud = df["fraud_flag"].sum()
n_total = len(df)

expected_overlap_rule = (n_rule_flagged * n_fraud) / n_total
expected_overlap_iso = (n_iso_flagged * n_fraud) / n_total

print(f"Rule-based flagged: {n_rule_flagged}, expected overlap by pure chance: {expected_overlap_rule:.2f}")
print(f"Isolation Forest flagged: {n_iso_flagged}, expected overlap by pure chance: {expected_overlap_iso:.2f}")

Rule-based flagged: 79, expected overlap by pure chance: 0.15
Isolation Forest flagged: 475, expected overlap by pure chance: 0.91


In [11]:
df["anomaly_score_raw"] = iso_forest.decision_function(X_scaled)  # lower = more anomalous

print(df.groupby("fraud_flag")["anomaly_score_raw"].describe())

               count      mean       std       min       25%       50%  \
fraud_flag                                                               
0           249520.0  0.148792  0.047824 -0.048820  0.114286  0.154700   
1              480.0  0.146157  0.049027  0.002823  0.109445  0.150172   

                 75%       max  
fraud_flag                      
0           0.187915  0.223778  
1           0.188613  0.222371  
